# Forward-Looking Huggett Model

Extends the Huggett (1993) setup by making the policy **forward-looking**: consumption at time $t$ depends on **$p_{t+1}$** (expected next-period price) in addition to $(b, y, z_t, r_t)$.

- **Internal trajectory**: A fixed sequence of aggregate states $z_0, z_1, \ldots, z_{T-1}$ (list of $z_t$).
- **$p$-trajectory**: A sequence $p_0, p_1, \ldots, p_T$. At time $t$ the policy uses **$p_{t+1}$** to choose $c_t$ (so agents look one period ahead).
- **Inner loop (N_p=10)**: For one fixed $z$-trajectory we (1) simulate with current $p$-trajectory and compute loss $L$; (2) gradient step on $\theta$; (3) set $p_t \leftarrow r_t$ (realized price) for the next inner iteration. Repeat 10 times.
- **Outer loop**: 100 epochs; each epoch samples (or reuses) $z$-trajectories and runs the inner loop.
- **Test & visualize**: Run on test trajectories and plot different $p_t$ paths (from inner iterations) and performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# Calibration: same as Huggett + forward-looking settings
beta, sigma = 0.96, 2.0
rho_y, nu_y = 0.6, 0.2
rho_z, nu_z = 0.9, 0.02
B, b_min = 0.0, -1.0
nb, b_max = 40, 50.0
ny, nr, nz = 3, 20, 30
r_min, r_max = 0.01, 0.06
c_min = 1e-3
e_trunc = 1e-3

# Forward-looking: inner loop N_p, outer epochs, trajectory length
N_p = 10
N_epoch_outer = 100
T_traj = 50
N_z_trajectories = 4
lr_ini = 1e-3

print("Forward-looking Huggett: N_p=%d, N_epoch=%d, T_traj=%d" % (N_p, N_epoch_outer, T_traj))

In [ ]:
# Grids and Tauchen (same as Huggett)
b_grid = np.linspace(b_min, b_max, nb)
r_grid = np.linspace(r_min, r_max, nr)

def tauchen_ar1(rho, sigma_innov, n_states, m=3, mean=0.0):
    std = sigma_innov / np.sqrt(1 - rho**2)
    if mean == 0:
        x_min, x_max = -m * std, m * std
    else:
        x_min = max(1e-6, mean - m * std)
        x_max = mean + m * std
    x_grid = np.linspace(x_min, x_max, n_states)
    step = (x_max - x_min) / (n_states - 1) if n_states > 1 else 1.0
    mu_i = (1 - rho) * mean + rho * x_grid
    z_lo = (x_grid - mu_i[:, None] + step / 2) / sigma_innov
    z_hi = (x_grid - mu_i[:, None] - step / 2) / sigma_innov
    P = np.zeros((n_states, n_states))
    P[:, 0] = norm.cdf(z_lo[:, 0])
    P[:, -1] = 1 - norm.cdf(z_hi[:, -1])
    if n_states > 2:
        P[:, 1:-1] = norm.cdf(z_lo[:, 1:-1]) - norm.cdf(z_hi[:, 1:-1])
    P = P / P.sum(axis=1, keepdims=True)
    return x_grid, P

y_grid, Ty = tauchen_ar1(rho_y, nu_y, ny, m=3, mean=1.0)
invariant_y = np.linalg.matrix_power(Ty.T, 200)[:, 0]
y_grid = y_grid / (y_grid @ invariant_y)

log_z_grid, Tz = tauchen_ar1(rho_z, nu_z, nz)
z_grid = np.exp(log_z_grid)
invariant_z = np.linalg.matrix_power(Tz.T, 200)[:, 0]
z_grid = z_grid / (z_grid @ invariant_z)

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu')
dtype = torch.float32
print('Device:', device)

nb_spg, nr_spg, nz_spg = 50, 10, 10
b_grid_t = torch.tensor(np.linspace(b_min, b_max, nb_spg), dtype=dtype, device=device)
iz_spg = np.linspace(0, nz-1, nz_spg, dtype=int)
z_grid_t = torch.tensor(z_grid[iz_spg], dtype=dtype, device=device)
r_grid_t = torch.tensor(np.linspace(r_min, r_max, nr_spg), dtype=dtype, device=device)
y_grid_t = torch.tensor(y_grid, dtype=dtype, device=device)
Ty_t = torch.tensor(Ty, dtype=dtype, device=device)
Tz_sub = Tz[np.ix_(iz_spg, iz_spg)] if len(iz_spg) <= len(z_grid) else Tz
Tz_sub = Tz_sub / Tz_sub.sum(axis=1, keepdims=True)
Tz_t = torch.tensor(Tz_sub, dtype=dtype, device=device)
nz_spg = Tz_t.shape[0]

def theta_to_consumption_grid(theta, c_min_val=1e-3):
    return torch.clamp(theta, min=c_min_val)

def init_theta(b_grid_t, y_grid_t, z_grid_t, r_grid_t, save_frac=0.2, c_min_val=1e-3):
    nb_t, ny_t = len(b_grid_t), len(y_grid_t)
    nz_t, nr_t = len(z_grid_t), len(r_grid_t)
    J = nb_t * ny_t
    b_flat = b_grid_t.repeat_interleave(ny_t)
    y_flat = y_grid_t.repeat(nb_t)
    cash = b_flat.view(J, 1, 1) * (1 + r_grid_t).view(1, 1, nr_t) + y_flat.view(J, 1, 1) * z_grid_t.view(1, nz_t, 1)
    c_grid = torch.clamp((1 - save_frac) * cash, min=c_min_val)
    return c_grid.view(nb_t, ny_t, nz_t, nr_t)

def u_torch(c, sig=sigma):
    c = torch.clamp(c, min=1e-10)
    if abs(sig - 1.0) < 1e-10:
        return torch.log(c)
    return (c ** (1 - sig)) / (1 - sig)

In [ ]:
def _G_to_mat_spg(G, nb_spg, ny):
    return G.view(nb_spg, ny) if G.dim() == 1 else G

def update_G_pi_direct(theta, G, iz, ir, b_grid_t, y_grid_t, z_grid_t, r_grid_t, Ty_t, nb_spg, ny, ip_consume=None):
    G = _G_to_mat_spg(G, nb_spg, ny)
    c = theta_to_consumption_grid(theta)
    ic = ip_consume if ip_consume is not None else ir
    c_val = c[:, :, iz, ic].ravel()
    b_next = (1 + r_grid_t[ir]) * b_grid_t.repeat_interleave(ny) + y_grid_t.repeat(nb_spg) * z_grid_t[iz] - c_val
    b_next = torch.clamp(b_next, b_grid_t[0], b_grid_t[-1])
    idx_hi = torch.searchsorted(b_grid_t, b_next).clamp(1, nb_spg - 1)
    idx_lo = idx_hi - 1
    w_hi = (b_next - b_grid_t[idx_lo]) / (b_grid_t[idx_hi] - b_grid_t[idx_lo]).clamp(min=1e-20)
    w_lo = 1.0 - w_hi
    eye = torch.eye(nb_spg, device=b_grid_t.device, dtype=b_grid_t.dtype)
    w_b = w_lo.unsqueeze(1) * eye[idx_lo] + w_hi.unsqueeze(1) * eye[idx_hi]
    M = w_b.view(nb_spg, ny, nb_spg).permute(2, 0, 1)
    Q = (M * G.unsqueeze(0)).sum(dim=1)
    G_new = Q @ Ty_t
    return G_new / (G_new.sum() + 1e-20)

def P_star_detach(theta, G, iz, b_grid_t, y_grid_t, z_grid_t, r_grid_t, ny, B=0.0, ip_consume=None):
    """Market-clearing r. If ip_consume is set (forward-looking), c uses that index only."""
    nr = len(r_grid_t)
    if nr == 1:
        return r_grid_t[0].detach()
    nb_b, G_mat = len(b_grid_t), _G_to_mat_spg(G, len(b_grid_t), ny)
    z_val = z_grid_t[iz]
    c_grid = theta_to_consumption_grid(theta)
    if ip_consume is not None:
        c_slice = c_grid[:, :, iz, ip_consume].reshape(nb_b * ny, 1).expand(-1, nr)
    else:
        c_slice = c_grid[:, :, iz, :].reshape(nb_b * ny, nr)
    b_flat = b_grid_t.repeat_interleave(ny)
    y_flat = y_grid_t.repeat(nb_b)
    resources = b_flat.unsqueeze(1) * (1 + r_grid_t).unsqueeze(0) + (y_flat * z_val).unsqueeze(1)
    b_next_all = (resources - c_slice).clamp(b_min, b_max)
    S_all = (G_mat.unsqueeze(2) * b_next_all.view(nb_b, ny, nr)).sum(dim=(0, 1))
    return r_grid_t[(S_all - B).abs().argmin().item()].detach()

def r_to_ir(r_val, r_grid_t):
    r = r_val.item() if torch.is_tensor(r_val) else float(r_val)
    grid = r_grid_t.cpu().numpy()
    idx = np.clip(np.searchsorted(grid, r, side='right') - 1, 0, len(grid) - 1)
    return idx

## Forward-looking objective: one (z_trajectory, p_trajectory)

At time $t$, policy uses **$p_{t+1}$** (index `ip_next`) to choose consumption; market clearing gives $r_t$ (for budget and G update). Returns $L$ and the list of realized `r_t` for updating the $p$-trajectory.

- **First inner iteration (k=0)**: No prior $p$-trajectory from simulation → assume $P_{t+1}=P_t$. Use $\\pi(b,y,r_t,r_t,z)$ (fourth argument = current $r_t$), same as in the baseline Huggett model.
- **Last period (t=T-1)**: No realized $p_{t+1}$ (no period $T$) → use $\\pi(b,y,r_t,r_t,z)$ as well.
- **Otherwise**: Use $p_{t+1}$ from the trajectory for the consumption index.

In [ ]:
def objective_one_trajectory(theta, z_trajectory, p_trajectory, G0, b_grid_t, y_grid_t, z_grid_t, r_grid_t,
                             Ty_t, Tz_t, nb_spg, ny, nz_spg, nr_spg, beta, e_trunc=1e-3, assume_pt1_equals_pt=False):
    """
    z_trajectory: list of iz of length T_traj (fixed z path).
    p_trajectory: list of length T_traj+1; p_trajectory[t+1] is used at t (expected next price).
    assume_pt1_equals_pt: if True (e.g. first inner iteration), use π[b][y][r][r][z] at all t (P_{t+1}=P_t).
    At the last period t=T-1 we always use π[b][y][r][r][z] since we have no realized p_{t+1}.
    Returns (L, r_realized_list) where r_realized_list has length T_traj.
    """
    T = len(z_trajectory)
    G = _G_to_mat_spg(G0.clone(), nb_spg, ny)
    L_n = torch.tensor(0.0, device=theta.device, dtype=theta.dtype)
    r_realized = []
    for t in range(T):
        if (beta ** t) < e_trunc:
            break
        iz = z_trajectory[t]
        is_last_period = (t == T - 1)
        # First training / first inner iter: assume P_{t+1}=P_t (π[b][y][r][r][z]). Last period: no p_{t+1} → same.
        use_same_r = assume_pt1_equals_pt or is_last_period
        if use_same_r:
            ip_consume_arg = None  # P_star finds r; we will use that ir for consumption (π[b][y][r][r][z])
        else:
            ip_consume_arg = p_trajectory[t + 1]
        r_t = P_star_detach(theta, G, iz, b_grid_t, y_grid_t, z_grid_t, r_grid_t, ny, B=B, ip_consume=ip_consume_arg)
        ir_t = r_to_ir(r_t, r_grid_t)
        r_realized.append(r_t.item() if torch.is_tensor(r_t) else r_t)
        ip_for_c_and_G = ir_t if use_same_r else ip_consume_arg
        c = theta_to_consumption_grid(theta)
        c_t = c[:, :, iz, ip_for_c_and_G]
        L_n = L_n + (beta ** t) * (G * u_torch(c_t)).sum()
        G = update_G_pi_direct(theta, G, iz, ir_t, b_grid_t, y_grid_t, z_grid_t, r_grid_t, Ty_t, nb_spg, ny, ip_consume=ip_for_c_and_G).detach()
    return L_n, r_realized

In [ ]:
def value_to_ir(p_val, r_grid_t):
    """Map scalar price p to grid index in [0, nr_spg-1]."""
    p = p_val if isinstance(p_val, float) else float(p_val)
    grid = r_grid_t.cpu().numpy()
    idx = np.clip(np.searchsorted(grid, p, side='right') - 1, 0, len(grid) - 1)
    return idx

def make_p_trajectory_indices(r_realized, r_grid_t, T_traj):
    """Convert list of realized r (length T_traj) to p_trajectory (length T_traj+1) as indices."""
    idxs = [value_to_ir(r, r_grid_t) for r in r_realized]
    idxs.append(idxs[-1] if idxs else 0)
    return idxs

## Inner loop (N_p times) and outer loop (100 epochs)

For each epoch we take one or more $z$-trajectories. For each $z$-trajectory we run the inner loop: simulate with current $p$-trajectory, gradient step on $\theta$, then set $p$-trajectory to the realized $r$ path. Repeat N_p times.

In [ ]:
def generate_z_trajectory(T, nz_spg, Tz_np):
    """Sample a z path from Tz (nz_spg x nz_spg). Returns list of iz of length T."""
    iz = np.random.randint(0, nz_spg)
    path = []
    for _ in range(T):
        path.append(iz)
        iz = np.random.choice(nz_spg, p=Tz_np[iz, :])
    return path

Tz_np = Tz_t.cpu().numpy()
G0_steady = torch.ones(nb_spg, ny, device=device, dtype=dtype) / (nb_spg * ny)
theta = init_theta(b_grid_t, y_grid_t, z_grid_t, r_grid_t)
theta = theta.requires_grad_(True)
optimizer = torch.optim.Adam([theta], lr=lr_ini)

loss_hist = []
p_trajectories_by_epoch = []
for epoch in range(N_epoch_outer):
    z_trajectory = generate_z_trajectory(T_traj, nz_spg, Tz_np)
    p_trajectory = [nr_spg // 2] * (T_traj + 1)
    epoch_losses = []
    epoch_pt = []
    for k in range(N_p):
        # First inner iteration: no p_trajectory from prior sim → assume P_{t+1}=P_t (π[b][y][r][r][z])
        assume_pt1 = (k == 0)
        L, r_realized = objective_one_trajectory(theta, z_trajectory, p_trajectory, G0_steady,
            b_grid_t, y_grid_t, z_grid_t, r_grid_t, Ty_t, Tz_t, nb_spg, ny, nz_spg, nr_spg, beta, e_trunc,
            assume_pt1_equals_pt=assume_pt1)
        optimizer.zero_grad()
        (-L).backward()
        optimizer.step()
        epoch_losses.append(L.item())
        epoch_pt.append(list(p_trajectory))
        p_trajectory = make_p_trajectory_indices(r_realized, r_grid_t, T_traj)
    loss_hist.append(np.mean(epoch_losses))
    p_trajectories_by_epoch.append(epoch_pt)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print("Epoch %d, mean L = %.4f" % (epoch + 1, loss_hist[-1]))

## Test on fixed trajectories and visualize

Run the inner loop on a few **fixed** $z$-trajectories (no outer epoch), record $p_t$ over inner iterations, and plot: (1) loss vs inner iteration; (2) $p_t$ (or $r_t$) over $t$ for different inner iterations.

In [ ]:
theta_test = theta.detach().clone().requires_grad_(True)
opt_test = torch.optim.Adam([theta_test], lr=lr_ini)

test_z_trajectories = [generate_z_trajectory(T_traj, nz_spg, Tz_np) for _ in range(N_z_trajectories)]
r_grid_np = r_grid_t.cpu().numpy()

test_results = []
for traj_idx, z_traj in enumerate(test_z_trajectories):
    p_traj = [nr_spg // 2] * (T_traj + 1)
    losses_inner = []
    r_paths_inner = []
    for k in range(N_p):
        assume_pt1 = (k == 0)
        L, r_realized = objective_one_trajectory(theta_test, z_traj, p_traj, G0_steady,
            b_grid_t, y_grid_t, z_grid_t, r_grid_t, Ty_t, Tz_t, nb_spg, ny, nz_spg, nr_spg, beta, e_trunc,
            assume_pt1_equals_pt=assume_pt1)
        opt_test.zero_grad()
        (-L).backward()
        opt_test.step()
        losses_inner.append(L.item())
        r_paths_inner.append(list(r_realized))
        p_traj = make_p_trajectory_indices(r_realized, r_grid_t, T_traj)
    test_results.append({"losses": losses_inner, "r_paths": r_paths_inner})
    print("Test trajectory %d: L first=%.4f, last=%.4f" % (traj_idx + 1, losses_inner[0], losses_inner[-1]))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

for traj_idx, res in enumerate(test_results):
    axs[0, 0].plot(range(N_p), res["losses"], label="z_traj %d" % (traj_idx + 1))
axs[0, 0].set_xlabel("Inner iteration k")
axs[0, 0].set_ylabel("L(θ)")
axs[0, 0].set_title("Loss over inner loop (different z-trajectories)")
axs[0, 0].legend()
axs[0, 0].grid(alpha=0.3)

t_axis = np.arange(T_traj)
for traj_idx, res in enumerate(test_results):
    for k in [0, N_p - 1]:
        axs[0, 1].plot(t_axis, res["r_paths"][k], alpha=0.7, label="z_traj%d, k=%d" % (traj_idx + 1, k))
axs[0, 1].set_xlabel("Time t")
axs[0, 1].set_ylabel("r_t (realized)")
axs[0, 1].set_title("Realized r_t: first vs last inner iteration")
axs[0, 1].legend(ncol=2, fontsize=8)
axs[0, 1].grid(alpha=0.3)

axs[1, 0].plot(loss_hist, color="tab:blue")
axs[1, 0].set_xlabel("Epoch")
axs[1, 0].set_ylabel("Mean L")
axs[1, 0].set_title("Training: mean L per epoch")
axs[1, 0].grid(alpha=0.3)

res0 = test_results[0]
for k in range(N_p):
    axs[1, 1].plot(t_axis, res0["r_paths"][k], alpha=0.5 + 0.5 * (k / max(N_p, 1)), label="k=%d" % k)
axs[1, 1].set_xlabel("Time t")
axs[1, 1].set_ylabel("r_t")
axs[1, 1].set_title("p_t trajectory evolution (z_traj 1, inner k=0..N_p-1)")
axs[1, 1].legend(ncol=2, fontsize=8)
axs[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()